In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from src.spark_session import get_spark
from src.config import CUSTOMERS_RAW_PATH, CUSTOMERS_SILVER_PATH

In [3]:
spark = get_spark("SilverCustomers")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/06 11:16:20 WARN Utils: Your hostname, Branimirs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.199 instead (on interface en0)
26/08/06 11:16:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/06 11:16:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applic

In [4]:
customers_raw_df = spark.read.option("header", True).option("inferSchema", True).csv(str(CUSTOMERS_RAW_PATH))
customers_raw_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [5]:
customers_raw_df.show(10, truncate=False)

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca               |SP            |
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|9790                    |sao bernardo do campo|SP            |
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29e8181a18229c7b0b2b5e|1151                    |sao paulo            |SP            |
|b2b6027bc5c5109e529d4dc6358b12c3|259dac757896d24d7702b9acbbff3f3c|8775                    |mogi das cruzes      |SP            |
|4f2d8ab171c80ec8364f7c12e35b23ad|345ecd01c38d18a9036ed96c73b8d066|13056                  

In [6]:
print("Rows: ", customers_raw_df.count())
print("Columns: ", len(customers_raw_df.columns))
print("Column names: ", customers_raw_df.columns)

Rows:  99441
Columns:  5
Column names:  ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [7]:
customer_row_count = customers_raw_df.count()

distinct_customer_id_row_count = (
    customers_raw_df.select("customer_id").distinct().count()
)

print("Rows: ", customer_row_count)
print("Distinct customer IDs: ", distinct_customer_id_row_count)
print(f"customer_id is unique: {customer_row_count == distinct_customer_id_row_count}")

Rows:  99441
Distinct customer IDs:  99441
customer_id is unique: True


In [8]:
distinct_unique_customer_count = (
    customers_raw_df.select("customer_unique_id").distinct().count()
)

print("Rows: ", customer_row_count)
print("Distinct unique customer IDs: ", distinct_unique_customer_count)
print(f"customer_id is unique: {customer_row_count == distinct_unique_customer_count}")

Rows:  99441
Distinct unique customer IDs:  96096
customer_id is unique: False


In [9]:
repeated_customers = (
    customers_raw_df
    .groupBy("customer_unique_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
)

repeated_customers.show(10, truncate=False)

+--------------------------------+-----+
|customer_unique_id              |count|
+--------------------------------+-----+
|8d50f5eadf50201ccdcedfb9e2ac8455|17   |
|3e43e6105506432c953e165fb2acf44c|9    |
|6469f99c1f9dfae7733b25662e7f1782|7    |
|1b6c7548a2a1f9037c1fd3ddfed95f33|7    |
|ca77025e7201e3b30c44b472ff346268|7    |
|47c1a3033b8b77b3ab6e109eb4d5fdf3|6    |
|f0e310a6839dce9de1638e0fe5ab282a|6    |
|dc813062e0fc23409cd255f7f53c7074|6    |
|12f5d6e1cbf93dafd9dcc19095df0b3d|6    |
|63cfc61cee11cbe306bff5857d00bfe4|6    |
+--------------------------------+-----+
only showing top 10 rows


In [10]:
customers_schema = StructType([
    StructField("customer_id", StringType(), nullable=False),
    StructField("customer_unique_id", StringType(), nullable=False),
    StructField("customer_zip_code_prefix", StringType(), nullable=True),
    StructField("customer_city", StringType(), nullable=True),
    StructField("customer_state", StringType(), nullable=True)
])

customers_typed_df = spark.read.option("header", True).schema(customers_schema).csv(str(CUSTOMERS_RAW_PATH))

In [11]:
customers_typed_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [12]:
customers_clean_df = (
    customers_typed_df
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("customer_unique_id").isNotNull())
    .withColumn(
        "customer_city",
        F.lower(F.trim(F.col("customer_city")))
    )
    .withColumn(
        "customer_state",
        F.lower(F.trim(F.col("customer_state")))
    )
    .dropDuplicates(["customer_id"])
)

In [13]:
customers_clean_df.show(10, truncate=False)

+--------------------------------+--------------------------------+------------------------+-------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city      |customer_state|
+--------------------------------+--------------------------------+------------------------+-------------------+--------------+
|00050bf6e01e69d5c0fd612f1bcfb69c|e3cf594a99e810f58af53ed4820f25e5|98700                   |ijui               |rs            |
|000598caf2ef4117407665ac33275130|7e0516b486e92ed3f3afdd6d1276cfbd|35540                   |oliveira           |mg            |
|0013cd8e350a7cc76873441e431dd5ee|334fed5abcee3aa96c13f1432703e1fd|03585                   |sao paulo          |sp            |
|0015bc9fd2d5395446143e8b215d7c75|490c854539b21598cfbbac518ca25788|12233                   |sao jose dos campos|sp            |
|001df1ee5c36767aa607001ab1a13a06|46b44ab325f78e5bb3dc0bbef1082082|01030                   |sao paulo   

In [14]:
customers_clean_df.groupBy("customer_state").count().orderBy(F.col("count").desc()).show(30)

+--------------+-----+
|customer_state|count|
+--------------+-----+
|            sp|41746|
|            rj|12852|
|            mg|11635|
|            rs| 5466|
|            pr| 5045|
|            sc| 3637|
|            ba| 3380|
|            df| 2140|
|            es| 2033|
|            go| 2020|
|            pe| 1652|
|            ce| 1336|
|            pa|  975|
|            mt|  907|
|            ma|  747|
|            ms|  715|
|            pb|  536|
|            pi|  495|
|            rn|  485|
|            al|  413|
|            se|  350|
|            to|  280|
|            ro|  253|
|            am|  148|
|            ac|   81|
|            ap|   68|
|            rr|   46|
+--------------+-----+



In [15]:
invalid_state_codes = customers_clean_df.filter(F.length(F.col("customer_state")) != 2)
invalid_state_codes.show(truncate=False)

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
+-----------+------------------+------------------------+-------------+--------------+



In [17]:
customers_clean_df.coalesce(2).write.mode("overwrite").parquet(str(CUSTOMERS_SILVER_PATH))

In [18]:
saved_customers_df = spark.read.parquet(str(CUSTOMERS_SILVER_PATH))
saved_customers_df.show(10, truncate=False)

+--------------------------------+--------------------------------+------------------------+----------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city   |customer_state|
+--------------------------------+--------------------------------+------------------------+----------------+--------------+
|000bf8121c3412d3057d32371c5d3395|1bc9b2dad6aefbfbc011508e34c8adfc|12335                   |jacarei         |sp            |
|00114026c1b7b52ab1773f317ef4880b|f4dc0a81a11d3d270ccf5a9c4b5b187b|22470                   |rio de janeiro  |rj            |
|0015f7887e2fde13ddaa7b8e385af919|866c923cde750dfc8cfbcf9d5ced0ee4|25903                   |mage            |rj            |
|001f6f1a5e902ad14e1f709a7215de11|c6b7dcd3718d1ad87f069d32a8566ce2|12460                   |campos do jordao|sp            |
|002348c1099e3229276c8ad7d4ddc702|934c19eeef04da89928f995df85cf3f8|13295                   |itupeva         |sp            |
